In [ ]:
# ============================================================
# FINAL HORIZON-SELECTION STUDY
# EXPERIMENT-GROUPED MULTICLASS FUSION-LSTM
#
# Horizons:
#   H = 5, 10, 15, 20, 25, 30 frames
#
# Input:
#   20 observations = t-19 ... t
#
# Target:
#   t + H
#
# Features:
#   10 instantaneous multimodal features
#   10 first-order temporal differences
#   Total = 20
#
# Evaluation:
#   5 experiment-grouped outer folds
#   2 complete validation experiments per outer fold
#   validation pair selected using development data only
#   10 random seeds
#
# Total model runs:
#   6 horizons × 5 folds × 10 seeds = 300 models
#
# IMPORTANT:
#   - Outer test experiment assignments are fixed across horizons
#   - No experiment overlap between train / validation / test
#   - StandardScaler fitted ONLY on training data
#   - Class weights derived ONLY from training targets
#   - Validation loss selects best model
#   - Test data evaluated only after model selection
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import random
import itertools
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 2. PATHS
# ============================================================

project_dir = Path.cwd()

outputs_dir = project_dir / "outputs"

data_path = (
    outputs_dir
    / "master_fusion_dataset_clean.csv"
)

save_dir = (
    outputs_dir
    / "Final_Grouped_Horizon_Selection_10Seeds"
)

save_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. GLOBAL SETTINGS
# ============================================================

sequence_length = 20

horizons = [
    5,
    10,
    15,
    20,
    25,
    30
]

epochs = 60
batch_size = 32
learning_rate = 0.001

hidden_size = 64
num_layers = 2
dropout = 0.2

num_classes = 4

outer_folds = 5

seeds = [
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51
]

class_names = {
    0: "Good",
    1: "Burr",
    2: "Flash-burr",
    3: "Surface-groove/void"
}

all_class_ids = [
    0,
    1,
    2,
    3
]


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 100)
print("FINAL GROUPED HORIZON-SELECTION STUDY")
print("MULTICLASS FUSION-LSTM")
print("=" * 100)

print("Device:", device)
print("Input sequence length:", sequence_length)
print("Horizons:", horizons)
print("Seeds:", seeds)

print(
    "Expected model runs:",
    len(horizons)
    * outer_folds
    * len(seeds)
)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

def set_seed(seed_value):

    random.seed(seed_value)
    np.random.seed(seed_value)

    torch.manual_seed(seed_value)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 5. LOAD DATA
# ============================================================

if not data_path.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{data_path}"
    )


df = pd.read_csv(
    data_path
)


df = (
    df
    .sort_values(
        [
            "exp_id",
            "frame_idx"
        ]
    )
    .reset_index(
        drop=True
    )
)


experiment_ids = sorted(
    df[
        "exp_id"
    ].unique()
)


print(
    "\nDataset shape:",
    df.shape
)

print(
    "Number of experiments:",
    len(experiment_ids)
)

print(
    "Experiments:",
    experiment_ids
)


# ============================================================
# 6. MULTICLASS TARGET
# ============================================================

def assign_defect_class(row):

    burr = float(

        row["Burrs_area_mm2"]

        if pd.notna(
            row["Burrs_area_mm2"]
        )

        else 0.0
    )


    flash = float(

        row["flash_burr_area_mm2"]

        if pd.notna(
            row["flash_burr_area_mm2"]
        )

        else 0.0
    )


    groove = float(

        row[
            "surface_groove_void_area_mm2"
        ]

        if pd.notna(
            row[
                "surface_groove_void_area_mm2"
            ]
        )

        else 0.0
    )


    defect_areas = {
        1: burr,
        2: flash,
        3: groove
    }


    dominant_class = max(
        defect_areas,
        key=defect_areas.get
    )


    if (
        defect_areas[
            dominant_class
        ]
        > 0
    ):

        return dominant_class


    return 0


df[
    "defect_class"
] = df.apply(
    assign_defect_class,
    axis=1
)


# ============================================================
# 7. DEFINE FEATURES
# ============================================================

base_features = [

    # Process-monitoring
    "sensor_force",
    "sensor_rpm",
    "sensor_torque",
    "sensor_temp",

    # Vision-derived
    "weld_width_mm",
    "weld_area_mm2",
    "Burrs_area_mm2",
    "flash_burr_area_mm2",
    "surface_groove_void_area_mm2",
    "total_defect_area_mm2"
]


for col in base_features:

    if col not in df.columns:

        raise ValueError(
            f"Missing feature: {col}"
        )


    df[col] = (
        df[col]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(
            0.0
        )
    )


# ============================================================
# 8. FIRST-ORDER TEMPORAL DIFFERENCES
# ============================================================
#
# Preserves current implementation:
#
# delta_x(t) = x(t) - x(t-1)
#
# ============================================================

for col in base_features:

    df[
        f"{col}_diff"
    ] = (
        df
        .groupby(
            "exp_id"
        )[col]
        .diff()
        .fillna(
            0.0
        )
    )


feature_cols = (

    base_features

    +

    [
        f"{col}_diff"
        for col
        in base_features
    ]
)


print(
    "\nNumber of input features:",
    len(feature_cols)
)


# ============================================================
# 9. FIX OUTER EXPERIMENT FOLDS
# ============================================================
#
# Very important:
#
# Outer experiment assignments are generated ONCE and reused
# for every forecasting horizon.
#
# This makes the horizon comparison fair.
#
# ============================================================

experiment_array = np.asarray(
    experiment_ids
)


dummy_X = np.zeros(
    (
        len(experiment_array),
        1
    )
)


outer_cv = GroupKFold(
    n_splits=outer_folds
)


fixed_outer_folds = []


for fold_number, (
    development_exp_indices,
    test_exp_indices

) in enumerate(

    outer_cv.split(
        dummy_X,
        groups=experiment_array
    ),

    start=1
):

    development_experiments = (

        experiment_array[
            development_exp_indices
        ].tolist()
    )


    test_experiments = (

        experiment_array[
            test_exp_indices
        ].tolist()
    )


    fixed_outer_folds.append({

        "fold":
            fold_number,

        "development_experiments":
            sorted(
                development_experiments
            ),

        "test_experiments":
            sorted(
                test_experiments
            )
    })


print(
    "\nFIXED OUTER FOLDS"
)


for fold_info in fixed_outer_folds:

    print(
        f"\nFold {fold_info['fold']}"
    )

    print(
        "Development:",
        fold_info[
            "development_experiments"
        ]
    )

    print(
        "Test:",
        fold_info[
            "test_experiments"
        ]
    )


# ============================================================
# 10. CREATE TEMPORAL SEQUENCES FOR ONE HORIZON
# ============================================================

def create_sequences_for_horizon(
    dataframe,
    future_horizon
):

    X_list = []
    y_list = []
    group_list = []
    metadata_list = []


    for exp_id, group in dataframe.groupby(
        "exp_id"
    ):

        group = (
            group
            .sort_values(
                "frame_idx"
            )
            .reset_index(
                drop=True
            )
        )


        X_exp = (
            group[
                feature_cols
            ]
            .values
            .astype(
                np.float32
            )
        )


        y_exp = (
            group[
                "defect_class"
            ]
            .values
            .astype(
                np.int64
            )
        )


        number_of_sequences = (

            len(group)

            - sequence_length

            - future_horizon

            + 1
        )


        if (
            number_of_sequences
            <= 0
        ):

            continue


        for start_idx in range(
            number_of_sequences
        ):

            input_start_idx = (
                start_idx
            )


            input_end_idx = (

                input_start_idx

                + sequence_length

                - 1
            )


            target_idx = (

                input_end_idx

                + future_horizon
            )


            X_sequence = (

                X_exp[
                    input_start_idx:
                    input_end_idx + 1
                ]
            )


            y_target = (
                y_exp[
                    target_idx
                ]
            )


            X_list.append(
                X_sequence
            )


            y_list.append(
                y_target
            )


            group_list.append(
                exp_id
            )


            metadata_list.append({

                "exp_id":
                    exp_id,

                "horizon":
                    future_horizon,

                "input_start_frame":
                    int(
                        group.loc[
                            input_start_idx,
                            "frame_idx"
                        ]
                    ),

                "input_end_frame":
                    int(
                        group.loc[
                            input_end_idx,
                            "frame_idx"
                        ]
                    ),

                "target_frame":
                    int(
                        group.loc[
                            target_idx,
                            "frame_idx"
                        ]
                    ),

                "true_class":
                    int(
                        y_target
                    )
            })


    X = np.asarray(
        X_list,
        dtype=np.float32
    )


    y = np.asarray(
        y_list,
        dtype=np.int64
    )


    groups = np.asarray(
        group_list
    )


    metadata = pd.DataFrame(
        metadata_list
    )


    return (
        X,
        y,
        groups,
        metadata
    )


# ============================================================
# 11. PYTORCH DATASET
# ============================================================

class SequenceDataset(
    Dataset
):

    def __init__(
        self,
        X,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )


    def __len__(self):

        return len(
            self.X
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )


# ============================================================
# 12. FUSION-LSTM
# ============================================================

class FusionLSTM(
    nn.Module
):

    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=2,
        num_classes=4,
        dropout=0.2
    ):

        super().__init__()


        self.lstm = nn.LSTM(

            input_size=
            input_size,

            hidden_size=
            hidden_size,

            num_layers=
            num_layers,

            batch_first=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            )
        )


        self.fc = nn.Sequential(

            nn.Linear(
                hidden_size,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        output, _ = (
            self.lstm(
                x
            )
        )


        final_hidden = (
            output[
                :,
                -1,
                :
            ]
        )


        return self.fc(
            final_hidden
        )


# ============================================================
# 13. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=
                all_class_ids,
                average=
                "macro",
                zero_division=
                0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=
                all_class_ids,
                average=
                "macro",
                zero_division=
                0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=
                all_class_ids,
                average=
                "macro",
                zero_division=
                0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                labels=
                all_class_ids,
                average=
                "weighted",
                zero_division=
                0
            )
    }


# ============================================================
# 14. CLASS COUNTS
# ============================================================

def get_class_counts(
    labels
):

    return np.bincount(
        labels,
        minlength=num_classes
    )


# ============================================================
# 15. TWO-EXPERIMENT VALIDATION SELECTION
# ============================================================
#
# For each horizon/fold, the validation pair is selected using
# DEVELOPMENT experiments only.
#
# Test experiments are never inspected.
#
# ============================================================

def choose_validation_pair(
    development_experiments,
    groups,
    labels
):

    candidate_rows = []


    for exp_a, exp_b in itertools.combinations(
        sorted(
            development_experiments
        ),
        2
    ):

        validation_mask = np.isin(
            groups,
            [
                exp_a,
                exp_b
            ]
        )


        validation_labels = (
            labels[
                validation_mask
            ]
        )


        counts = get_class_counts(
            validation_labels
        )


        represented_classes = int(
            np.sum(
                counts > 0
            )
        )


        minimum_class_support = int(
            np.min(
                counts
            )
        )


        total_samples = int(
            counts.sum()
        )


        if (
            total_samples > 0
            and
            np.all(
                counts > 0
            )
        ):

            proportions = (
                counts.astype(
                    float
                )
                /
                total_samples
            )


            balance_score = (

                num_classes

                /
                np.sum(
                    1.0
                    /
                    proportions
                )
            )

        else:

            balance_score = (
                0.0
            )


        candidate_rows.append({

            "exp_a":
                exp_a,

            "exp_b":
                exp_b,

            "represented_classes":
                represented_classes,

            "minimum_class_support":
                minimum_class_support,

            "balance_score":
                balance_score,

            "validation_samples":
                total_samples,

            "Good":
                int(
                    counts[0]
                ),

            "Burr":
                int(
                    counts[1]
                ),

            "Flash_burr":
                int(
                    counts[2]
                ),

            "Surface_groove_void":
                int(
                    counts[3]
                )
        })


    candidate_df = pd.DataFrame(
        candidate_rows
    )


    candidate_df = (

        candidate_df

        .sort_values(

            by=[
                "represented_classes",
                "minimum_class_support",
                "balance_score",
                "validation_samples",
                "exp_a",
                "exp_b"
            ],

            ascending=[
                False,
                False,
                False,
                False,
                True,
                True
            ]
        )

        .reset_index(
            drop=True
        )
    )


    best_pair = [

        candidate_df.loc[
            0,
            "exp_a"
        ],

        candidate_df.loc[
            0,
            "exp_b"
        ]
    ]


    return (
        best_pair,
        candidate_df
    )


# ============================================================
# 16. STANDARDIZATION
# ============================================================

def standardize_fold(
    X_train,
    X_validation,
    X_test
):

    n_features = (
        X_train.shape[
            2
        ]
    )


    scaler = StandardScaler()


    scaler.fit(

        X_train.reshape(
            -1,
            n_features
        )
    )


    def transform(
        X
    ):

        original_shape = (
            X.shape
        )


        return (

            scaler

            .transform(
                X.reshape(
                    -1,
                    n_features
                )
            )

            .reshape(
                original_shape
            )

            .astype(
                np.float32
            )
        )


    return (
        transform(
            X_train
        ),
        transform(
            X_validation
        ),
        transform(
            X_test
        ),
        scaler
    )


# ============================================================
# 17. TRAIN ONE MODEL
# ============================================================

def train_model(
    X_train,
    y_train,
    X_validation,
    y_validation,
    horizon,
    fold,
    seed_value
):

    set_seed(
        seed_value
    )


    train_dataset = SequenceDataset(
        X_train,
        y_train
    )


    validation_dataset = SequenceDataset(
        X_validation,
        y_validation
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed_value
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=
        batch_size,

        shuffle=
        True,

        generator=
        generator,

        num_workers=
        0
    )


    validation_loader = DataLoader(

        validation_dataset,

        batch_size=
        batch_size,

        shuffle=
        False,

        num_workers=
        0
    )


    # --------------------------------------------------------
    # CLASS WEIGHTS FROM TRAINING ONLY
    # --------------------------------------------------------

    class_counts = get_class_counts(
        y_train
    )


    class_weights = (

        len(
            y_train
        )

        /

        (
            num_classes

            *
            np.maximum(
                class_counts,
                1
            )
        )
    )


    class_weight_tensor = torch.tensor(

        class_weights,

        dtype=
        torch.float32,

        device=
        device
    )


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model = FusionLSTM(

        input_size=
        X_train.shape[
            2
        ],

        hidden_size=
        hidden_size,

        num_layers=
        num_layers,

        num_classes=
        num_classes,

        dropout=
        dropout

    ).to(
        device
    )


    criterion = nn.CrossEntropyLoss(
        weight=
        class_weight_tensor
    )


    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=
        learning_rate
    )


    best_validation_loss = (
        np.inf
    )


    best_epoch = None


    best_state = None


    history = []


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(
        1,
        epochs + 1
    ):

        # -------------------------
        # TRAIN
        # -------------------------

        model.train()


        training_loss_sum = (
            0.0
        )

        training_count = (
            0
        )


        for (
            X_batch,
            y_batch

        ) in train_loader:

            X_batch = (
                X_batch.to(
                    device
                )
            )


            y_batch = (
                y_batch.to(
                    device
                )
            )


            optimizer.zero_grad()


            logits = model(
                X_batch
            )


            loss = criterion(
                logits,
                y_batch
            )


            loss.backward()


            optimizer.step()


            n_batch = (
                y_batch.size(
                    0
                )
            )


            training_loss_sum += (

                loss.item()

                *
                n_batch
            )


            training_count += (
                n_batch
            )


        average_training_loss = (

            training_loss_sum

            /
            training_count
        )


        # -------------------------
        # VALIDATION
        # -------------------------

        model.eval()


        validation_loss_sum = (
            0.0
        )

        validation_count = (
            0
        )


        with torch.no_grad():

            for (
                X_batch,
                y_batch

            ) in validation_loader:

                X_batch = (
                    X_batch.to(
                        device
                    )
                )


                y_batch = (
                    y_batch.to(
                        device
                    )
                )


                logits = model(
                    X_batch
                )


                loss = criterion(
                    logits,
                    y_batch
                )


                n_batch = (
                    y_batch.size(
                        0
                    )
                )


                validation_loss_sum += (

                    loss.item()

                    *
                    n_batch
                )


                validation_count += (
                    n_batch
                )


        average_validation_loss = (

            validation_loss_sum

            /
            validation_count
        )


        history.append({

            "epoch":
                epoch,

            "training_loss":
                average_training_loss,

            "validation_loss":
                average_validation_loss
        })


        if (
            average_validation_loss
            <
            best_validation_loss
        ):

            best_validation_loss = (
                average_validation_loss
            )


            best_epoch = (
                epoch
            )


            best_state = deepcopy(
                model.state_dict()
            )


        if (

            epoch == 1

            or
            epoch % 20 == 0

            or
            epoch == epochs
        ):

            print(

                f"H={horizon:02d} | "
                f"Fold={fold} | "
                f"Seed={seed_value} | "
                f"Epoch={epoch:02d} | "

                f"Train="
                f"{average_training_loss:.4f} | "

                f"Val="
                f"{average_validation_loss:.4f}"
            )


    model.load_state_dict(
        best_state
    )


    return (
        model,
        best_epoch,
        best_validation_loss,
        pd.DataFrame(
            history
        )
    )


# ============================================================
# 18. TEST MODEL
# ============================================================

def evaluate_model(
    model,
    X_test,
    y_test
):

    dataset = SequenceDataset(
        X_test,
        y_test
    )


    loader = DataLoader(

        dataset,

        batch_size=
        batch_size,

        shuffle=
        False,

        num_workers=
        0
    )


    model.eval()


    all_true = []
    all_pred = []


    with torch.no_grad():

        for (
            X_batch,
            y_batch

        ) in loader:

            X_batch = (
                X_batch.to(
                    device
                )
            )


            logits = model(
                X_batch
            )


            prediction = torch.argmax(
                logits,
                dim=1
            )


            all_true.extend(
                y_batch.numpy()
            )


            all_pred.extend(

                prediction

                .cpu()

                .numpy()
            )


    y_true = np.asarray(
        all_true
    )


    y_pred = np.asarray(
        all_pred
    )


    return (
        calculate_metrics(
            y_true,
            y_pred
        ),
        y_true,
        y_pred
    )


# ============================================================
# 19. STORAGE
# ============================================================

all_fold_seed_results = []

all_horizon_seed_results = []

sequence_count_rows = []


# ============================================================
# 20. HORIZON LOOP
# ============================================================

for future_horizon in horizons:

    print(
        "\n"
        +
        "#" * 100
    )

    print(
        f"FORECASTING HORIZON = "
        f"{future_horizon} FRAMES"
    )

    print(
        f"Approx. time = "
        f"{future_horizon / 30.0:.3f} s"
    )

    print(
        "#" * 100
    )


    horizon_dir = (

        save_dir

        /
        f"H_{future_horizon}"
    )


    horizon_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Create sequences
    # --------------------------------------------------------

    (
        X_all,
        y_all,
        groups_all,
        metadata_all

    ) = create_sequences_for_horizon(

        df,

        future_horizon
    )


    print(
        "Total sequences:",
        len(
            y_all
        )
    )


    class_counts_all = get_class_counts(
        y_all
    )


    sequence_count_rows.append({

        "horizon_frames":
            future_horizon,

        "horizon_seconds":
            future_horizon / 30.0,

        "total_sequences":
            len(
                y_all
            ),

        "Good":
            int(
                class_counts_all[0]
            ),

        "Burr":
            int(
                class_counts_all[1]
            ),

        "Flash_burr":
            int(
                class_counts_all[2]
            ),

        "Surface_groove_void":
            int(
                class_counts_all[3]
            )
    })


    # --------------------------------------------------------
    # Pooled predictions per seed
    # --------------------------------------------------------

    pooled_true_by_seed = {

        seed_value: []

        for seed_value
        in seeds
    }


    pooled_pred_by_seed = {

        seed_value: []

        for seed_value
        in seeds
    }


    # ========================================================
    # OUTER FOLD LOOP
    # ========================================================

    for fold_info in fixed_outer_folds:

        fold_number = (
            fold_info[
                "fold"
            ]
        )


        development_experiments = (

            fold_info[
                "development_experiments"
            ]
        )


        test_experiments = (

            fold_info[
                "test_experiments"
            ]
        )


        # ----------------------------------------------------
        # Development / test masks
        # ----------------------------------------------------

        development_mask = np.isin(

            groups_all,

            development_experiments
        )


        test_mask = np.isin(

            groups_all,

            test_experiments
        )


        development_indices = np.where(
            development_mask
        )[0]


        test_indices = np.where(
            test_mask
        )[0]


        # ----------------------------------------------------
        # Select validation pair using development only
        # ----------------------------------------------------

        (
            validation_experiments,
            validation_candidate_df

        ) = choose_validation_pair(

            development_experiments,

            groups_all[
                development_indices
            ],

            y_all[
                development_indices
            ]
        )


        print(
            f"\nH={future_horizon} | "
            f"Fold {fold_number}"
        )

        print(
            "Validation:",
            validation_experiments
        )

        print(
            "Test:",
            test_experiments
        )


        # ----------------------------------------------------
        # Get validation/train masks
        # ----------------------------------------------------

        validation_mask_global = np.isin(

            groups_all,

            validation_experiments
        )


        training_experiments = [

            exp_id

            for exp_id
            in development_experiments

            if exp_id
            not in
            validation_experiments
        ]


        training_mask_global = np.isin(

            groups_all,

            training_experiments
        )


        training_indices = np.where(
            training_mask_global
        )[0]


        validation_indices = np.where(
            validation_mask_global
        )[0]


        # ----------------------------------------------------
        # Experiment leakage safeguards
        # ----------------------------------------------------

        assert set(
            training_experiments
        ).isdisjoint(
            set(
                validation_experiments
            )
        )


        assert set(
            training_experiments
        ).isdisjoint(
            set(
                test_experiments
            )
        )


        assert set(
            validation_experiments
        ).isdisjoint(
            set(
                test_experiments
            )
        )


        # ----------------------------------------------------
        # Arrays
        # ----------------------------------------------------

        X_train_raw = (
            X_all[
                training_indices
            ]
        )

        y_train = (
            y_all[
                training_indices
            ]
        )


        X_validation_raw = (
            X_all[
                validation_indices
            ]
        )

        y_validation = (
            y_all[
                validation_indices
            ]
        )


        X_test_raw = (
            X_all[
                test_indices
            ]
        )

        y_test = (
            y_all[
                test_indices
            ]
        )


        # ----------------------------------------------------
        # Training-only scaler
        # ----------------------------------------------------

        (
            X_train,
            X_validation,
            X_test,
            scaler

        ) = standardize_fold(

            X_train_raw,

            X_validation_raw,

            X_test_raw
        )


        # ----------------------------------------------------
        # Save validation-pair candidates
        # ----------------------------------------------------

        fold_dir = (

            horizon_dir

            /
            f"fold_{fold_number}"
        )


        fold_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        validation_candidate_df.to_csv(

            fold_dir
            /
            "validation_pair_candidates.csv",

            index=False
        )


        # ====================================================
        # SEED LOOP
        # ====================================================

        for seed_value in seeds:

            (
                model,
                best_epoch,
                best_validation_loss,
                history_df

            ) = train_model(

                X_train,

                y_train,

                X_validation,

                y_validation,

                future_horizon,

                fold_number,

                seed_value
            )


            (
                metrics,
                y_true,
                y_pred

            ) = evaluate_model(

                model,

                X_test,

                y_test
            )


            # ------------------------------------------------
            # Store fold result
            # ------------------------------------------------

            all_fold_seed_results.append({

                "horizon_frames":
                    future_horizon,

                "horizon_seconds":
                    future_horizon
                    /
                    30.0,

                "fold":
                    fold_number,

                "seed":
                    seed_value,

                "training_experiments":
                    ", ".join(
                        training_experiments
                    ),

                "validation_experiments":
                    ", ".join(
                        validation_experiments
                    ),

                "test_experiments":
                    ", ".join(
                        test_experiments
                    ),

                "n_train":
                    len(
                        y_train
                    ),

                "n_validation":
                    len(
                        y_validation
                    ),

                "n_test":
                    len(
                        y_test
                    ),

                "best_epoch":
                    best_epoch,

                "best_validation_loss":
                    best_validation_loss,

                "accuracy":
                    metrics[
                        "accuracy"
                    ],

                "macro_precision":
                    metrics[
                        "macro_precision"
                    ],

                "macro_recall":
                    metrics[
                        "macro_recall"
                    ],

                "macro_f1":
                    metrics[
                        "macro_f1"
                    ],

                "weighted_f1":
                    metrics[
                        "weighted_f1"
                    ]
            })


            # ------------------------------------------------
            # Pooled OOF
            # ------------------------------------------------

            pooled_true_by_seed[
                seed_value
            ].extend(
                y_true
            )


            pooled_pred_by_seed[
                seed_value
            ].extend(
                y_pred
            )


            # ------------------------------------------------
            # Save history
            # ------------------------------------------------

            history_df.to_csv(

                fold_dir
                /
                (
                    f"training_history_"
                    f"seed_{seed_value}.csv"
                ),

                index=False
            )


    # ========================================================
    # POOLED RESULT FOR EACH SEED AT THIS HORIZON
    # ========================================================

    print(
        "\n"
        +
        "=" * 100
    )

    print(
        f"POOLED RESULTS FOR H="
        f"{future_horizon}"
    )

    print(
        "=" * 100
    )


    for seed_value in seeds:

        pooled_true = np.asarray(

            pooled_true_by_seed[
                seed_value
            ]
        )


        pooled_pred = np.asarray(

            pooled_pred_by_seed[
                seed_value
            ]
        )


        metrics = calculate_metrics(

            pooled_true,

            pooled_pred
        )


        all_horizon_seed_results.append({

            "horizon_frames":
                future_horizon,

            "horizon_seconds":
                future_horizon
                /
                30.0,

            "seed":
                seed_value,

            "total_oof_sequences":
                len(
                    pooled_true
                ),

            "accuracy":
                metrics[
                    "accuracy"
                ],

            "macro_precision":
                metrics[
                    "macro_precision"
                ],

            "macro_recall":
                metrics[
                    "macro_recall"
                ],

            "macro_f1":
                metrics[
                    "macro_f1"
                ],

            "weighted_f1":
                metrics[
                    "weighted_f1"
                ]
        })


        print(
            f"Seed {seed_value}: "
            f"Accuracy="
            f"{100 * metrics['accuracy']:.2f}% | "
            f"Macro F1="
            f"{100 * metrics['macro_f1']:.2f}%"
        )


# ============================================================
# 21. SAVE RAW RESULTS
# ============================================================

fold_seed_df = pd.DataFrame(

    all_fold_seed_results
)


fold_seed_df.to_csv(

    save_dir
    /
    "all_horizon_fold_seed_results.csv",

    index=False
)


horizon_seed_df = pd.DataFrame(

    all_horizon_seed_results
)


horizon_seed_df.to_csv(

    save_dir
    /
    "pooled_horizon_metrics_all_10_seeds.csv",

    index=False
)


sequence_count_df = pd.DataFrame(

    sequence_count_rows
)


sequence_count_df.to_csv(

    save_dir
    /
    "horizon_sequence_counts.csv",

    index=False
)


# ============================================================
# 22. FINAL HORIZON SUMMARY
# ============================================================

summary_rows = []


metric_columns = [

    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1"
]


for future_horizon in horizons:

    horizon_subset = (

        horizon_seed_df[
            horizon_seed_df[
                "horizon_frames"
            ]
            ==
            future_horizon
        ]
    )


    row = {

        "horizon_frames":
            future_horizon,

        "horizon_seconds":
            future_horizon
            /
            30.0,

        "total_sequences":
            int(
                horizon_subset[
                    "total_oof_sequences"
                ]
                .iloc[0]
            )
    }


    for metric in metric_columns:

        values = (

            horizon_subset[
                metric
            ]
            .values

            * 100.0
        )


        mean_value = (
            np.mean(
                values
            )
        )


        sd_value = (
            np.std(
                values,
                ddof=1
            )
        )


        ci_half = (

            1.96

            * sd_value

            /
            np.sqrt(
                len(
                    values
                )
            )
        )


        row[
            f"{metric}_mean"
        ] = (
            mean_value
        )


        row[
            f"{metric}_sd"
        ] = (
            sd_value
        )


        row[
            f"{metric}_ci95_lower"
        ] = (
            mean_value
            -
            ci_half
        )


        row[
            f"{metric}_ci95_upper"
        ] = (
            mean_value
            +
            ci_half
        )


    summary_rows.append(
        row
    )


final_summary_df = pd.DataFrame(

    summary_rows
)


final_summary_df.to_csv(

    save_dir
    /
    "FINAL_horizon_selection_summary.csv",

    index=False
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "FINAL HORIZON-SELECTION SUMMARY"
)

print(
    "=" * 120
)


columns_to_print = [

    "horizon_frames",

    "horizon_seconds",

    "total_sequences",

    "accuracy_mean",

    "accuracy_sd",

    "macro_f1_mean",

    "macro_f1_sd"
]


print(

    final_summary_df[
        columns_to_print
    ]

    .to_string(
        index=False
    )
)


# ============================================================
# 23. DETERMINE BEST HORIZON BY MACRO F1
# ============================================================
#
# IMPORTANT:
# Do not choose automatically based on Macro F1 alone for the
# manuscript. Prediction lead time and stability should also
# be considered.
#
# This is printed only as an objective reference.
#
# ============================================================

best_macro_f1_index = (

    final_summary_df[
        "macro_f1_mean"
    ]
    .idxmax()
)


best_row = (

    final_summary_df.loc[
        best_macro_f1_index
    ]
)


print(
    "\nHighest mean Macro F1:"
)


print(
    f"Horizon = "
    f"{int(best_row['horizon_frames'])} frames "
    f"({best_row['horizon_seconds']:.3f} s)"
)


print(
    f"Macro F1 = "
    f"{best_row['macro_f1_mean']:.2f} "
    f"± "
    f"{best_row['macro_f1_sd']:.2f}%"
)


print(
    f"Accuracy = "
    f"{best_row['accuracy_mean']:.2f} "
    f"± "
    f"{best_row['accuracy_sd']:.2f}%"
)


# ============================================================
# 24. PLOT — ACCURACY VS HORIZON
# ============================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.errorbar(

    final_summary_df[
        "horizon_frames"
    ],

    final_summary_df[
        "accuracy_mean"
    ],

    yerr=
    final_summary_df[
        "accuracy_sd"
    ],

    marker="o",

    capsize=4
)


plt.xlabel(
    "Forecasting Horizon (frames)"
)

plt.ylabel(
    "Accuracy (%)"
)

plt.title(
    "Experiment-Grouped Accuracy vs Forecasting Horizon"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()


plt.savefig(

    save_dir
    /
    "horizon_accuracy_mean_sd.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 25. PLOT — MACRO F1 VS HORIZON
# ============================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.errorbar(

    final_summary_df[
        "horizon_frames"
    ],

    final_summary_df[
        "macro_f1_mean"
    ],

    yerr=
    final_summary_df[
        "macro_f1_sd"
    ],

    marker="o",

    capsize=4
)


plt.xlabel(
    "Forecasting Horizon (frames)"
)

plt.ylabel(
    "Macro F1-score (%)"
)

plt.title(
    "Experiment-Grouped Macro F1 vs Forecasting Horizon"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()


plt.savefig(

    save_dir
    /
    "horizon_macro_f1_mean_sd.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ============================================================
# 26. FINAL INFORMATION
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "HORIZON-SELECTION STUDY COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nResults saved to:"
)

print(
    save_dir
)


print(
    "\nHorizons evaluated:",
    horizons
)


print(
    "Outer folds:",
    outer_folds
)


print(
    "Seeds per horizon:",
    len(
        seeds
    )
)


print(
    "Total model runs:",
    len(
        horizons
    )
    *
    outer_folds
    *
    len(
        seeds
    )
)


print(
    "\nMethodological safeguards:"
)


print(
    "1. Same outer experiment folds are used for every horizon."
)


print(
    "2. Complete welding experiments are held out for testing."
)


print(
    "3. Two complete validation experiments are selected from development data only."
)


print(
    "4. No experiment overlap exists between training, validation, and testing."
)


print(
    "5. Temporal sequences never cross experiment boundaries."
)


print(
    "6. StandardScaler is fitted only using training data."
)


print(
    "7. Class weights are calculated only using training targets."
)


print(
    "8. Lowest validation loss selects the model."
)


print(
    "9. Test data are evaluated only after model selection."
)


print(
    "10. Ten random seeds quantify training stochasticity."
)


print(
    "\nDone."
)